# 01 — Full generation run for one model (with checkpointing)

Runs **all 1,319 GSM8K test questions** and **all TriviaQA validation questions (no context)**
on ONE model, scores every answer, and saves everything.

**The same notebook is used for every model.** Set `SIZE` (model size) and `TAG` in cell 2:
- `"fp16"`  → the original Qwen2.5-Instruct model of that size (run this first)
- `"rtn4"`  → RTN-4bit model
- `"gptq4"` → GPTQ-4bit act-order model (C4 calibration)
- `"gptq4math"` → GPTQ-4bit act-order model calibrated on GSM8K train (`CALIB = "math"` in the quantize notebook)

**Inputs to attach (+ Add Input → Your Work):** the quantize notebook of that size
(`02_quantize_model` for 0.5B/3B; for 1.5B: 02-Quantize and 02_GPTQ_CHECK). Not needed for `"fp16"`.

Files of the 1.5B study keep their old names (`gen_fp16_gsm8k.jsonl`); other sizes get the size in front
(`gen_3b-fp16_gsm8k.jsonl`), so files of different sizes can never be mixed up.

**Run it with Save Version → Save & Run All (Commit)**, GPU T4 x2, Internet On.

**Checkpointing:** every batch is written to the results file immediately. If a run stops
(timeout, crash), open that failed version, attach it as an Input (**+ Add Input → Your Work**),
and run again: cell 4 copies the partial file back and the run continues from where it stopped.

**Saved files (in `/kaggle/working`, kept in the version's Output tab):**
- `gen_<TAG>_gsm8k.jsonl` — one line per question: question, gold, model output, prediction, correct, confidence
- `gen_<TAG>_triviaqa.jsonl` — same for TriviaQA
- `gen_<TAG>_summary.json` — accuracies, settings, library versions

In [ ]:
# 1. Versions and GPU
import sys, torch, transformers, datasets
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU!")

In [ ]:
# 2. Settings  -- change SIZE and TAG for each run
SIZE = "1.5b"                      # "0.5b", "1.5b" or "3b"
TAG  = "fp16"                      # "fp16", "rtn4", "gptq4" or "gptq4math"

HF_ID = {"0.5b": "Qwen/Qwen2.5-0.5B-Instruct", "1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
         "3b": "Qwen/Qwen2.5-3B-Instruct", "7b": "Qwen/Qwen2.5-7B-Instruct"}[SIZE]
SHORT = f"qwen2.5-{SIZE}"
MODELS = {                         # where each model comes from
    "fp16":  HF_ID,                              # Hugging Face
    "rtn4":  f"{SHORT}-rtn4-g128",               # folder from the quantize notebook (attach as Input)
    "gptq4": f"{SHORT}-gptq4-g128-c4-act",       # folder from the quantize notebook (attach as Input)
    "gptq4math": f"{SHORT}-gptq4-g128-math-act", # quantize notebook with CALIB = "math"
}
FTAG = TAG if SIZE == "1.5b" else f"{SIZE}-{TAG}"   # name used in the output files
N_GSM      = None    # None = all 1,319 test questions
N_TQA      = None    # None = all TriviaQA validation questions
SEED       = 42      # only used if N_TQA is set (random subset)
BATCH_GSM  = 16
BATCH_TQA  = 64
MAX_NEW_GSM = 1024   # raised from 512 after the pilot
MAX_NEW_TQA = 32
OUT_DIR    = "/kaggle/working"

In [ ]:
# 3. Helper functions: answer extraction and scoring (same as the pilot + accent fix)
import re
import string
import unicodedata
from fractions import Fraction


# ---------- GSM8K ----------
def gsm8k_gold(answer_text):
    """Gold answer is the number after '####'."""
    return answer_text.split("####")[-1].strip().replace(",", "")


def last_boxed(text):
    """Content of the last \\boxed{...}, handling nested braces."""
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    i = start + len("\\boxed{")
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c)
        i += 1
    return "".join(out) if depth == 0 else None


def to_number(s):
    """Turn '1,234', '$12.50', '50%', '\\frac{11}{2}', '11/2' into a float. None if impossible."""
    if s is None:
        return None
    s = s.strip()
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    m = re.search(r"\\frac\{\s*(-?[\d.]+)\s*\}\{\s*(-?[\d.]+)\s*\}", s)
    if m:
        try:
            return float(Fraction(m.group(1)) / Fraction(m.group(2)))
        except (ValueError, ZeroDivisionError):
            return None
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    s = s.replace(",", "").replace("$", "").replace("%", "").replace("\\", "")
    m = re.search(r"-?\d+(?:\.\d+)?\s*/\s*-?\d+(?:\.\d+)?", s)
    if m:
        try:
            a, b = m.group(0).split("/")
            return float(Fraction(a.strip()) / Fraction(b.strip()))
        except (ValueError, ZeroDivisionError):
            return None
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else None


def gsm8k_pred(output):
    """Prediction = last \\boxed{} if present, otherwise the last number in the text."""
    boxed = last_boxed(output)
    if boxed is not None:
        return boxed, "boxed"
    nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", output)
    return (nums[-1] if nums else None), "last_number"


def gsm8k_correct(pred, gold):
    p, g = to_number(pred), to_number(gold)
    return p is not None and g is not None and abs(p - g) < 1e-4


# ---------- TriviaQA ----------
def normalize_answer(s):
    """Standard SQuAD/TriviaQA normalization: lowercase, no accents, no punctuation, no articles."""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))   # Río -> Rio
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def trivia_pred(output):
    """Keep the first line and drop a leading 'Answer:'."""
    line = output.strip().split("\n")[0].strip()
    line = re.sub(r"^(final\s+)?answer\s*:\s*", "", line, flags=re.I)
    return line


def trivia_scores(pred, aliases):
    """em: normalized prediction equals an alias.
    contains: an alias appears as whole words inside a short prediction (<= 10 words)."""
    p = normalize_answer(pred)
    norm_aliases = {normalize_answer(a) for a in aliases if a.strip()}
    em = p in norm_aliases
    contains = len(p.split()) <= 10 and any(
        a and re.search(r"(^| )" + re.escape(a) + r"( |$)", p) for a in norm_aliases
    )
    return em, contains

In [ ]:
# 4. Checkpoint helpers
import os, glob, shutil, json, time

def restore(name):
    """If `name` is not in /kaggle/working but an earlier run of this notebook is attached
    as an Input, copy it here so the run continues instead of starting over."""
    dst = os.path.join(OUT_DIR, name)
    if os.path.exists(dst):
        return True
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        return False
    if os.path.isdir(hits[0]):
        shutil.copytree(hits[0], dst)
    else:
        shutil.copy(hits[0], dst)
    print("restored", name, "from", hits[0])
    return True

def read_jsonl(path):
    rows = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        rows.append(json.loads(line))
                    except json.JSONDecodeError:   # last line cut by a crash
                        pass
    return rows

def model_path(tag):
    src = MODELS[tag]
    if tag == "fp16":
        return src
    hits = [h for h in glob.glob(f"/kaggle/input/**/{src}", recursive=True) if os.path.isdir(h)]
    if not hits:
        raise FileNotFoundError(f"Attach the quantize notebook's output as an Input: folder '{src}' not found")
    return hits[0]

In [ ]:
# 5. Load data (same order every time, so ids match across models)
from datasets import load_dataset

gsm = load_dataset("openai/gsm8k", "main", split="test")
tqa = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
if N_GSM:
    gsm = gsm.select(range(N_GSM))
if N_TQA:
    tqa = tqa.shuffle(seed=SEED).select(range(N_TQA))

gsm_items = [{"id": i, "question": x["question"], "gold": gsm8k_gold(x["answer"])}
             for i, x in enumerate(gsm)]
tqa_items, seen = [], set()
DEDUP = SIZE != "1.5b"              # the 1.5B study ran every row (17,944, some questions twice); keep that exact
for x in tqa:                       # order there so the batches match. Other sizes: keep the first copy only.
    if DEDUP and x["question_id"] in seen:
        continue
    seen.add(x["question_id"])
    tqa_items.append({"id": x["question_id"], "question": x["question"], "gold": x["answer"]["value"],
                      "aliases": x["answer"]["aliases"] + [x["answer"]["value"]]})
print(len(gsm_items), "GSM8K questions |", len(tqa_items), "TriviaQA questions")

In [ ]:
# 6. Load the model
from transformers import AutoTokenizer, AutoModelForCausalLM
path = model_path(TAG)
print("model:", TAG, "->", path)
tok = AutoTokenizer.from_pretrained(path)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16,
                                             device_map={"": 0}).eval()
eos = model.generation_config.eos_token_id
EOS_IDS = set(eos if isinstance(eos, list) else [eos])

In [ ]:
# 7. Prompts + greedy generation with confidence (same as the pilot)
from transformers import LogitsProcessor, LogitsProcessorList

def gsm_prompt(q):
    msgs = [{"role": "user",
             "content": q + "\nPlease reason step by step, and put your final answer within \\boxed{}."}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def tqa_prompt(q):
    msgs = [{"role": "user",
             "content": "Answer the following question with only the answer, in a few words.\n"
                        "Question: " + q}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

class RecordLogprob(LogitsProcessor):
    # greedy picks the most likely token -> its log-prob is the max log-softmax
    def __init__(self):
        self.steps = []
    def __call__(self, input_ids, scores):
        self.steps.append(torch.log_softmax(scores.float(), dim=-1).max(dim=-1).values)
        return scores

@torch.no_grad()
def generate_batch(prompts, max_new):
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    rec = RecordLogprob()
    seqs = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                          temperature=None, top_p=None, top_k=None, repetition_penalty=1.0,
                          logits_processor=LogitsProcessorList([rec]),
                          pad_token_id=tok.pad_token_id)
    gen = seqs[:, enc["input_ids"].shape[1]:]
    logp = torch.stack(rec.steps, dim=1).float().cpu()
    out = []
    for j in range(gen.shape[0]):
        ids = gen[j].tolist()
        n = next((k + 1 for k, t in enumerate(ids) if t in EOS_IDS), len(ids))
        lp = logp[j, :n]
        out.append({"output": tok.decode(ids[:n], skip_special_tokens=True),
                    "n_tokens": n,
                    "hit_limit": n == max_new and ids[n - 1] not in EOS_IDS,
                    "mean_logprob": lp.mean().item(),
                    "min_logprob": lp.min().item()})
    return out

In [ ]:
# 8. Run one dataset with checkpointing (appends each batch to the file right away)
def score_gsm(it, r):
    pred, how = gsm8k_pred(r["output"])
    return {"pred": pred, "extracted_by": how, "correct": gsm8k_correct(pred, it["gold"])}

def score_tqa(it, r):
    pred = trivia_pred(r["output"])
    em, contains = trivia_scores(pred, it["aliases"])
    return {"pred": pred, "em": em, "correct": contains}     # main score = contains

def run(name, items, make_prompt, max_new, batch_size, score_fn):
    fname = f"gen_{FTAG}_{name}.jsonl"
    restore(fname)
    path = os.path.join(OUT_DIR, fname)
    prev = read_jsonl(path)
    with open(path, "w") as f:                 # rewrite only complete lines
        for r in prev:
            f.write(json.dumps(r) + "\n")
    done = {r["id"] for r in prev}
    todo = [it for it in items if it["id"] not in done]
    print(f"{name}: {len(done)} already done, {len(todo)} to go")
    t0 = time.time()
    with open(path, "a") as f:
        for b, i in enumerate(range(0, len(todo), batch_size)):
            batch = todo[i:i + batch_size]
            outs = generate_batch([make_prompt(it["question"]) for it in batch], max_new)
            for it, r in zip(batch, outs):
                row = {k: v for k, v in it.items() if k != "aliases"}
                row.update(score_fn(it, r)); row.update(r); row["model"] = TAG; row["size"] = SIZE
                f.write(json.dumps(row) + "\n")
            f.flush()
            if b % 10 == 0 or i + batch_size >= len(todo):
                k = min(i + batch_size, len(todo))
                el = time.time() - t0
                print(f"  {k}/{len(todo)} | {el/60:.1f} min | ETA {el/k*(len(todo)-k)/60:.1f} min")
    return read_jsonl(path)

gsm_rows = run("gsm8k", gsm_items, gsm_prompt, MAX_NEW_GSM, BATCH_GSM, score_gsm)

In [ ]:
# 9. TriviaQA
tqa_rows = run("triviaqa", tqa_items, tqa_prompt, MAX_NEW_TQA, BATCH_TQA, score_tqa)

In [ ]:
# 10. Summary (also saved as a file)
def acc(rows, key="correct"):
    return sum(bool(r[key]) for r in rows) / max(len(rows), 1)

summary = {
    "model": TAG, "size": SIZE, "path": path,
    "gsm8k": {"n": len(gsm_rows), "accuracy": acc(gsm_rows),
              "boxed": sum(r["extracted_by"] == "boxed" for r in gsm_rows),
              "hit_limit": sum(r["hit_limit"] for r in gsm_rows)},
    "triviaqa": {"n": len(tqa_rows), "accuracy_contains": acc(tqa_rows),
                 "accuracy_em": acc(tqa_rows, "em")},
    "settings": {"max_new_gsm": MAX_NEW_GSM, "max_new_tqa": MAX_NEW_TQA,
                 "decoding": "greedy, repetition_penalty=1.0", "dtype": "float16"},
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "datasets": datasets.__version__},
}
with open(f"{OUT_DIR}/gen_{FTAG}_summary.json", "w") as f:
    json.dump(summary, f, indent=1)
print(json.dumps(summary, indent=1))